# 03. Modelling & Benchmarking
Pada tahap ini kita menguji 4 model *Machine Learning* yang kuat terhadap data berukuran kecil:
1. Logistic Regression
2. Support Vector Machine (SVM)
3. Random Forest
4. XGBoost

Evaluasi dilakukan dengan teknik **Stratified K-Fold Cross Validation** pada data *Train*, lalu diakhiri dengan evaluasi performa pada data *Test* (unseen data) menggunakan metrik *Classification Report* dan *ROC AUC*.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc

# Konfigurasi path
BASE_DIR = Path.cwd().parent.resolve()
RESULTS_PATH = str(BASE_DIR / 'results')

# Load data yang sudah dipreprocess
train_df = pd.read_csv(f'{RESULTS_PATH}/train_preprocessed.csv')
test_df = pd.read_csv(f'{RESULTS_PATH}/test_preprocessed.csv')

# Pisahkan X dan y
X_train = train_df.drop(columns=['Success'])
y_train = train_df['Success']

X_test = test_df.drop(columns=['Success'])
y_test = test_df['Success']

### 1. Inisialisasi Model

In [9]:
# Kita set probability=True pada SVM agar bisa menghitung probabilitas kurva ROC
# Pada XGBoost, kita menggunakan parameter regulasi yang lebih aman agar mencegah overfit di data kecil
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', max_depth=3, reg_lambda=1)
}

### 2. K-Fold Cross Validation (Pada Data Training)
Mengevaluasi performa rata-rata secara internal membagi data *training* ke dalam 5 lipatan (*fold*).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    # Evaluasi menggunakan skor Akurasi rata-rata
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results.append({
        'Model': name,
        'CV Accuracy Mean': scores.mean(),
        'CV Accuracy Std': scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values(by='CV Accuracy Mean', ascending=False)
display(cv_df)

# Visualisasi CV Score dengan Plotly
fig = px.bar(
    cv_df, 
    x='Model', 
    y='CV Accuracy Mean', 
    error_y='CV Accuracy Std', 
    text_auto='.3f', 
    title='Perbandingan Skor Cross-Validation pada Data Training'
)
fig.show()
fig.write_html(f'{RESULTS_PATH}/cv_benchmarking.html')

,Model,CV Accuracy Mean,CV Accuracy Std
1,SVM,0.940000,0.027080
2,Random Forest,0.933333,0.018257
3,XGBoost,0.926667,0.022608
0,Logistic Regression,0.876667,0.044222


### 3. Evaluasi Classification Report pada Data Testing (Unseen Data)

In [11]:
roc_curves = {}  # Untuk menyimpan data sumbu X, Y kurva ROC

print("================ CLASSIFICATION REPORTS ================")
for name, model in models.items():
    # Train model pada seluruh data X_train
    model.fit(X_train, y_train)
    
    # Lakukan Prediksi
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Print Laporan
    print(f"\n---> Model: {name} <+++")
    print(classification_report(y_test, y_pred))
    
    auc_score = roc_auc_score(y_test, y_prob)
    print(f"ROC AUC Score: {auc_score:.4f}")
    
    # Simpan titik koordinat kurva ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': auc_score}
    
print("=========================================================")

================ CLASSIFICATION REPORTS ================

---> Model: Logistic Regression <+++
              precision    recall  f1-score   support

           0       1.00      0.71      0.83        38
           1       0.52      1.00      0.69        12

    accuracy                           0.78        50
   macro avg       0.76      0.86      0.76        50
weighted avg       0.89      0.78      0.80        50

ROC AUC Score: 0.9627

---> Model: SVM <+++
              precision    recall  f1-score   support

           0       0.91      0.82      0.86        38
           1       0.56      0.75      0.64        12

    accuracy                           0.80        50
   macro avg       0.74      0.78      0.75        50
weighted avg       0.83      0.80      0.81        50

ROC AUC Score: 0.9013

---> Model: Random Forest <+++
              precision    recall  f1-score   support

           0       0.89      0.82      0.85        38
           1       0.53      0.67      0.59 

### 4. Visualisasi Kurva ROC-AUC

In [12]:
fig_roc = go.Figure()
# Tambahkan garis referensi tebakan acak (random guess)
fig_roc.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)

for name, roc in roc_curves.items():
    fig_roc.add_trace(go.Scatter(
        x=roc['fpr'], 
        y=roc['tpr'], 
        name=f"{name} (AUC={roc['auc']:.3f})",
        mode='lines'
    ))

fig_roc.update_layout(
    title='Perbandingan ROC Curves pada Data Testing',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain='domain'),
    width=700, height=500
)
fig_roc.show()
fig_roc.write_html(f'{RESULTS_PATH}/roc_curve_benchmarking.html')

### 5. Area Hyperparameter Tuning
**[JANGAN DI-RUN DULU]**
Setelah Anda melihat dan menganalisis metrik `classification_report` dan skor ROC AUC di atas, pilih **1 atau 2 model terbaik** untuk dioptimasi lebih lanjut.

Di bawah ini adalah contoh *template* kode untuk melakukan **GridSearchCV**. Hapus *comment* block `"""` jika Anda sudah yakin model mana yang akan di-*tuning*.

In [13]:
"""
from sklearn.model_selection import GridSearchCV

# ==========================================
# CONTOH TUNING UNTUK RANDOM FOREST
# ==========================================
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 3, 5, 10],
    'min_samples_split': [2, 5, 10]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best Params Random Forest:", grid_rf.best_params_)
print("Best CV Accuracy Score:", grid_rf.best_score_)

# ==========================================
# CONTOH TUNING UNTUK SVM
# ==========================================
svm_params = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}
grid_svm = GridSearchCV(SVC(probability=True, random_state=42), svm_params, cv=5, scoring='accuracy', n_jobs=-1)
grid_svm.fit(X_train, y_train)
print("\nBest Params SVM:", grid_svm.best_params_)
print("Best CV Accuracy Score:", grid_svm.best_score_)
"""

'\nfrom sklearn.model_selection import GridSearchCV\n\n# ==========================================\n# CONTOH TUNING UNTUK RANDOM FOREST\n# ==========================================\nrf_params = {\n    \'n_estimators\': [50, 100, 200],\n    \'max_depth\': [None, 3, 5, 10],\n    \'min_samples_split\': [2, 5, 10]\n}\ngrid_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring=\'accuracy\', n_jobs=-1)\ngrid_rf.fit(X_train, y_train)\nprint("Best Params Random Forest:", grid_rf.best_params_)\nprint("Best CV Accuracy Score:", grid_rf.best_score_)\n\n# ==========================================\n# CONTOH TUNING UNTUK SVM\n# ==========================================\nsvm_params = {\n    \'C\': [0.1, 1, 10, 100],\n    \'gamma\': [\'scale\', \'auto\', 0.01, 0.1],\n    \'kernel\': [\'rbf\', \'linear\']\n}\ngrid_svm = GridSearchCV(SVC(probability=True, random_state=42), svm_params, cv=5, scoring=\'accuracy\', n_jobs=-1)\ngrid_svm.fit(X_train, y_train)\nprint("\nBest 

### 6. Export Semua Model & Model Terbaik
Mengekstrak dan menyimpan seluruh model *benchmark* dan otomatis mendeteksi model dengan skor **ROC AUC tertinggi** pada data `Test`.

In [14]:
# Kita simpan model-model ini di sub-folder khusus agar rapi
MODELS_DIR = f"{RESULTS_PATH}/models"
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

best_auc = 0
best_model_name = ""

print("Menyimpan semua model (Benchmark awal)...")
for name, model in models.items():
    # Simpan model
    file_name = f"{MODELS_DIR}/{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(model, file_name)
    print(f"[✓] Tersimpan: {file_name}")
    
    # Pencarian model dengan AUC tertinggi dari dictionary roc_curves (dihitung di Cell Bagian 3)
    auc_score = roc_curves[name]['auc']
    if auc_score > best_auc:
        best_auc = auc_score
        best_model_name = name

print("\n==============================================")
print(f"🏆 BEST MODEL OTOMATIS (By AUC): {best_model_name} (AUC: {best_auc:.4f})")
print("==============================================")

# Simpan salinan 'best_model.pkl'
best_model_file = f"{MODELS_DIR}/best_model.pkl"
joblib.dump(models[best_model_name], best_model_file)
print(f"Salinan model terbaik telah dibuat di: {best_model_file}")

print("\n(Catatan: Jika Anda melakukan Tuning Hyperparameter di atas, pastikan Anda juga menimpa (overwrite) best_model.pkl dengan model hasil tuning tersebut!)")

Menyimpan semua model (Benchmark awal)...
[✓] Tersimpan: D:\naufalarizq\ipb\lomba\infest\results/models/logistic_regression.pkl
[✓] Tersimpan: D:\naufalarizq\ipb\lomba\infest\results/models/svm.pkl
[✓] Tersimpan: D:\naufalarizq\ipb\lomba\infest\results/models/random_forest.pkl
[✓] Tersimpan: D:\naufalarizq\ipb\lomba\infest\results/models/xgboost.pkl

🏆 BEST MODEL OTOMATIS (By AUC): Logistic Regression (AUC: 0.9627)
Salinan model terbaik telah dibuat di: D:\naufalarizq\ipb\lomba\infest\results/models/best_model.pkl

(Catatan: Jika Anda melakukan Tuning Hyperparameter di atas, pastikan Anda juga menimpa (overwrite) best_model.pkl dengan model hasil tuning tersebut!)
